# 04 — Demand Forecasting Engine: Global ML Forecaster & Quantile Intervals

**Deliverable D3 Companion Notebook**

This notebook details the training, validation, and out-of-sample backtesting of the **Global `HistGradientBoostingRegressor`** demand forecasting engine against the Seasonal-Naive baseline.

### Core Highlights:
- **Global Statistical Strength:** A single model trained across 5,000 SKUs with 45 causal features.
- **Rule 3 & Rule 5 Compliance:** Strictly temporal feature engineering (lags, rolling stats, Fourier seasonality); zero future leakage.
- **Uncertainty Intervals:** Dual quantile regressors at $\alpha=0.10$ and $\alpha=0.90$ delivering 80% prediction intervals.
- **WAPE Reduction:** ML earns production deployment by beating the baseline on 3/3 folds.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go

from foresight.config import load_settings

settings = load_settings()
artifacts = settings.artifacts_dir
metrics_dir = artifacts / 'metrics'
forecasts_dir = artifacts / 'forecasts'

cv_path = metrics_dir / 'cv_summary.json'
cv_summary = json.loads(cv_path.read_text(encoding='utf-8'))
print("Loaded cross-validation summary.")

Loaded cross-validation summary.


## 1. Machine Learning vs. Baseline Scorecard

We compare the ML model's WAPE, Bias, and MAE directly against the Seasonal-Naive baseline across all 3 rolling-origin folds.

In [2]:
comp = cv_summary.get('comparison', {})
ml_summary = cv_summary.get('ml', {})

rows = []
for b_fold, m_fold in zip(cv_summary['folds'], ml_summary['folds']):
    delta = m_fold['wape'] - b_fold['wape']
    rel_red = (b_fold['wape'] - m_fold['wape']) / b_fold['wape']
    rows.append({
        'Fold': b_fold['fold'],
        'Train End': b_fold['train_end_week'],
        'Valid Weeks': b_fold['valid_weeks'],
        'Baseline WAPE': f"{b_fold['wape']:.2%}",
        'ML WAPE': f"{m_fold['wape']:.2%}",
        'Delta (pp)': f"{delta*100:+.2f}%",
        'Relative Reduction': f"{rel_red:.2%}",
        'ML Bias': f"{m_fold['bias']:+.4f}"
    })

scorecard_df = pd.DataFrame(rows)
display(scorecard_df)

print(f"\nPooled Baseline WAPE: {comp.get('baseline_overall_wape', 0):.2%}")
print(f"Pooled ML Forecaster WAPE: {comp.get('ml_overall_wape', 0):.2%}")
print(f"Relative WAPE Reduction: {comp.get('relative_reduction', 0):.2%}")
print(f"Beats Baseline: {comp.get('beats_baseline')} (3 out of 3 folds improved)")

,Fold,Train End,Valid Weeks,Baseline WAPE,ML WAPE,Delta (pp),Relative Reduction,ML Bias
0,1,2024-W52,8,38.01%,30.33%,-7.67%,20.19%,+0.0547
1,2,2025-W16,8,35.29%,26.96%,-8.33%,23.62%,+0.0476
2,3,2025-W40,12,39.05%,33.28%,-5.77%,14.78%,-0.0274



Pooled Baseline WAPE: 37.76%
Pooled ML Forecaster WAPE: 30.81%
Relative WAPE Reduction: 0.00%
Beats Baseline: True (3 out of 3 folds improved)


## 2. 80% Prediction Intervals Evaluation

Quantile regressors trained at $\alpha = 0.10$ and $\alpha = 0.90$ produce asymmetric, empirical prediction intervals.

In [3]:
coverage = ml_summary.get('interval_coverage', {})
print(f"Nominal Interval Coverage: {coverage.get('nominal', 0.8):.1%}")
print(f"Empirical Coverage on Fold 3: {coverage.get('coverage', 0):.2%}")
print(f"Mean Prediction Interval Width: {coverage.get('mean_width', 0):.2f} units")
print(f"Evaluation Horizon: {coverage.get('n_rows', 0):,} SKU-weeks")

Nominal Interval Coverage: 80.0%
Empirical Coverage on Fold 3: 77.63%
Mean Prediction Interval Width: 19.46 units
Evaluation Horizon: 60,000 SKU-weeks


## 3. Production Forecast Visualization

We load the out-of-sample forward 8-week production forecast (`artifacts/forecasts/forecast_8w.parquet`) and plot the point prediction and 80% confidence interval band.

In [4]:
prod_forecast = pd.read_parquet(forecasts_dir / 'forecast_8w.parquet')
sample_sku = prod_forecast['sku_id'].iloc[0]
sku_fc = prod_forecast[prod_forecast['sku_id'] == sample_sku].sort_values('year_week')

fig = go.Figure()

# Upper and Lower bound ribbon
fig.add_trace(go.Scatter(
    x=list(sku_fc['year_week']) + list(sku_fc['year_week'])[::-1],
    y=list(sku_fc['forecast_q90']) + list(sku_fc['forecast_q10'])[::-1],
    fill='toself',
    fillcolor='rgba(31, 119, 180, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo="skip",
    showlegend=True,
    name='80% Prediction Interval'
))

# Point forecast
fig.add_trace(go.Scatter(
    x=sku_fc['year_week'],
    y=sku_fc['forecast'],
    mode='lines+markers',
    line=dict(color='rgb(31, 119, 180)', width=3),
    name='ML Point Forecast'
))

fig.update_layout(
    title=f"8-Week Production Forecast with 80% Prediction Interval for {sample_sku}",
    xaxis_title="Forward Year-Week",
    yaxis_title="Forecast Demand (Units)",
    template="plotly_white"
)
fig.show()

## 4. Deliverable D3 Gate Verification

- **ML WAPE:** 30.81% (vs Baseline 37.76%) — Gate Passed!
- **Model weights:** Serialized in `artifacts/models/global_forecaster.joblib`
- **Audit memo:** `reports/model_evaluation.md`.